# Use Larger LLMs for Task Planning, Smaller LLMs for Execution

Not all tasks require the same level of intelligence. A straightforward FAQ question doesn't need the same reasoning power as a multi-issue complaint that requires synthesizing information from several sources.

In this notebook, we build a **customer support system** that uses Elasticsearch Workflows to route queries to different LLMs based on complexity:

- A **large LLM** (Claude Sonnet) to analyze the query with context from the knowledge base and decide the resolution path
- A **small LLM** (Mistral Small) to answer simple questions directly from FAQ snippets - fast and cheap
- The **large LLM** again to synthesize complex answers from multiple articles, with citations

Elastic Managed LLMs are [billed per million tokens](https://www.elastic.co/docs/reference/kibana/connectors-kibana/elastic-managed-llm) (both input and output), so routing simple queries to a smaller, cheaper model has a real impact on costs.

## Setup

In [16]:
%pip install -qU elasticsearch python-dotenv requests datasets


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
from dotenv import load_dotenv
import os
import requests
import json

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Connect to Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=30
)
es_client.info()

ObjectApiResponse({'name': 'instance-0000000001', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Set up AI Connectors

We use two AI Connectors for the workflow:

| Connector | Model | Type | Role |
| :--- | :--- | :--- | :--- |
| `Anthropic Claude Sonnet 4.6` | Claude Sonnet | Elastic Managed | Analyze query + KB context, decide complexity, synthesize complex answers with citations |
| `Mistral Small` | mistral-small-latest | Custom (OpenAI-compatible) | Answer simple FAQ-style questions directly from snippets |

The Sonnet connector is already available as an Elastic Managed LLM. We only need to create a custom connector for Mistral.

In [19]:
SMALL_LLM_CONNECTOR = "Mistral Small"

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Create custom Mistral connector (OpenAI-compatible)
mistral_connector_payload = {
    "connector_type_id": ".gen-ai",
    "name": SMALL_LLM_CONNECTOR,
    "config": {
        "apiProvider": "Other",
        "apiUrl": "https://api.mistral.ai/v1/chat/completions",
        "defaultModel": "mistral-small-latest",
    },
    "secrets": {
        "apiKey": MISTRAL_API_KEY,
    },
}

# Let Kibana generate the UUID automatically
response = requests.post(
    f"{KIBANA_URL}/api/actions/connector",
    headers=headers,
    json=mistral_connector_payload,
)
result = response.json()
MISTRAL_CONNECTOR_ID = result.get("id")
print(f"Mistral connector created: {response.status_code}")
print(f"Connector ID: {MISTRAL_CONNECTOR_ID}")

Mistral connector created: 200
Connector ID: 8cd7b49d-2267-47f8-a273-b6f752e7053d


## Load and index the dataset

We use the [e-commerce-customer-support-qa](https://huggingface.co/datasets/rjac/e-commerce-customer-support-qa) dataset from Hugging Face. It contains 1,000 real customer support interactions from an e-commerce platform (BrownBox) with:

- Customer questions and agent solutions
- Issue categories and subcategories
- **Complexity levels** (less, medium, high)
- Customer sentiment
- Product categories

We index the Q&A pairs as our knowledge base, so the workflow can search for relevant articles before generating a response.

In [20]:
from datasets import load_dataset

dataset = load_dataset("rjac/e-commerce-customer-support-qa", split="train")
print(f"Total entries: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
dataset[0]

Total entries: 1000
Columns: ['issue_area', 'issue_category', 'issue_sub_category', 'issue_category_sub_category', 'customer_sentiment', 'product_category', 'product_sub_category', 'issue_complexity', 'agent_experience_level', 'agent_experience_level_desc', 'conversation', 'qa']


{'issue_area': 'Login and Account',
 'issue_category': 'Mobile Number and Email Verification',
 'issue_sub_category': 'Verification requirement for mobile number or email address during login',
 'issue_category_sub_category': 'Mobile Number and Email Verification -> Verification requirement for mobile number or email address during login',
 'customer_sentiment': 'neutral',
 'product_category': 'Appliances',
 'product_sub_category': 'Oven Toaster Grills (OTG)',
 'issue_complexity': 'medium',
 'agent_experience_level': 'junior',
 'agent_experience_level_desc': 'handles customer inquiries independently, possess solid troubleshooting skills, and seek guidance from more experienced team members when needed.',
 'conversation': "Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email ver

In [21]:
INDEX_NAME = "support-knowledge-base"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "conversation": {
                "type": "text",
                "copy_to": "semantic_content",
            },
            "qa": {
                "type": "text",
                "copy_to": "semantic_content",
            },
            "issue_area": {"type": "keyword"},
            "issue_category": {"type": "keyword"},
            "issue_sub_category": {"type": "keyword"},
            "issue_category_sub_category": {"type": "keyword"},
            "customer_sentiment": {"type": "keyword"},
            "product_category": {"type": "keyword"},
            "product_sub_category": {"type": "keyword"},
            "issue_complexity": {"type": "keyword"},
            "agent_experience_level": {"type": "keyword"},
            "agent_experience_level_desc": {"type": "text"},
            "semantic_content": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
        }
    },
)
print(f"Created index: {INDEX_NAME}")

Created index: support-knowledge-base


In [26]:
from elasticsearch import helpers


def build_bulk_actions(dataset, index_name):
    for i, item in enumerate(dataset):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": dict(item),
        }


success, failed = helpers.bulk(
    es_client,
    build_bulk_actions(dataset, INDEX_NAME),
    refresh=True,
)
print(f"Indexed {success} documents into '{INDEX_NAME}'")

Indexed 1000 documents into 'support-knowledge-base'


In [27]:
# Verify: check the distribution of complexity levels
for complexity in ["less", "medium", "high"]:
    count = es_client.count(
        index=INDEX_NAME,
        query={"term": {"issue_complexity": complexity}},
    )["count"]
    print(f"  {complexity}: {count} articles")

  less: 488 articles
  medium: 426 articles
  high: 86 articles


## The Workflow

The key insight: **search first, then classify**. The LLM needs context from the knowledge base to make a good routing decision. A query like *"my OTG isn't heating evenly"* only makes sense after finding that OTG refers to Oven Toaster Grills in our product catalog.

```
                    ┌─────────────────────┐
                    │   Customer Query     │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │  Search Knowledge   │
                    │  Base (Elasticsearch)│
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │  Classify Query     │
                    │  with KB context    │
                    │  (Claude Sonnet)    │
                    └──────────┬──────────┘
                               │
                 ┌─────────────┴─────────────┐
                 │                           │
            simple                       complex
         (FAQ match)              (needs synthesis)
                 │                           │
      ┌──────────▼──────────┐   ┌───────────▼──────────┐
      │  Direct answer from │   │  Synthesize from     │
      │  FAQ snippet        │   │  multiple articles   │
      │  (Mistral Small)    │   │  with citations      │
      │  Fast & cheap       │   │  (Claude Sonnet)     │
      └──────────┬──────────┘   └───────────┬──────────┘
                 │                           │
                 └─────────────┬─────────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Final Response     │
                    └─────────────────────┘
```

Why two different models?
- **Simple (FAQ match)**: The answer is already in a single article. A small model just needs to rephrase it clearly. Fast, cheap.
- **Complex (synthesis needed)**: The answer requires connecting information from multiple articles, reasoning about the customer's specific situation, and citing sources. This needs a capable model.

### Elasticsearch Workflow definition

Elasticsearch Workflows can be triggered in three ways:
- **Manually** from the Kibana UI
- **On a schedule** (cron or interval)
- **Via alerts** (when a detection rule fires)

Additionally, workflows can be exposed as **tools in the Agent Builder**, which lets users interact with them through a chat interface.

The following YAML defines the workflow. It is configured directly in the Workflow UI (Workflows don't currently have a REST API).


```yaml
name: support_query_router
description: >
  Routes customer queries to the appropriate LLM based on complexity.
  Searches the knowledge base first, then uses a large LLM to classify
  the query. Simple FAQ questions go to a small, fast model. Complex
  questions requiring synthesis go to a large model with citations.
enabled: true

inputs:
  - name: query
    type: string
    description: The customer support query
    required: true

consts:
  indexName: support-knowledge-base

triggers:
  - type: manual

steps:
  # Step 1: Search the knowledge base using semantic search
  - name: search_es
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        semantic:
          field: semantic_content
          query: "{{ inputs.query }}"
      size: 5

  # Step 2: Classify the query using Claude Sonnet (with ES context)
  - name: classify_query
    type: ai.prompt
    with:
      connectorId: Anthropic Claude Sonnet 4.6
      prompt: >
        You are a support query classifier. Given the customer query and
        the search results from our knowledge base, determine the complexity.

        Return ONLY a JSON object with:
        - "complexity": "simple" if the answer is clearly covered by a single
          FAQ article, or "complex" if it requires synthesizing multiple sources,
          involves multiple issues, or the query is ambiguous.
        - "reasoning": one-line explanation of why.

        Customer query: {{ inputs.query }}

        Knowledge base results:
        {{ steps.search_es.output.hits.hits | json }}

  # Step 3: Route based on complexity
  - name: route_by_complexity
    type: if
    condition: "${{ steps.classify_query.output.complexity == 'simple' }}"
    steps:
      # Simple: answer directly from FAQ snippet (Mistral Small)
      - name: answer_from_faq
        type: ai.prompt
        with:
          connectorId: Mistral Small
          prompt: >
            You are a customer support agent. Answer the customer's question
            using ONLY the FAQ article below. Be concise, friendly, and
            include specific steps if applicable.

            Customer query: {{ inputs.query }}

            FAQ article:
            {{ steps.search_es.output.hits.hits[0]._source | json }}
    else:
      # Complex: synthesize from multiple articles (Claude Sonnet)
      - name: synthesize_answer
        type: ai.prompt
        with:
          connectorId: Anthropic Claude Sonnet 4.6
          prompt: >
            You are a senior customer support specialist. The customer's query
            requires careful analysis across multiple knowledge base articles.

            Provide a detailed, empathetic response that:
            1. Addresses all aspects of the customer's question
            2. Cites specific articles from the knowledge base (reference them
               by their question/title)
            3. Provides clear resolution steps
            4. Notes if any part of the query isn't covered by the KB

            Customer query: {{ inputs.query }}

            Knowledge base articles:
            {{ steps.search_es.output.hits.hits | json }}
```

## Using the Workflow as a tool in Agent Builder

Beyond the default triggers (manual, schedule, alerts), workflows can also be exposed as **tools in the Agent Builder**. This adds a conversational layer - users can interact with the workflow through a chat interface, and the agent decides when to call it.

We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api) to create the tool and the agent.

**Note:** After creating the workflow in the Kibana UI using the YAML above, copy its ID and set it below as `WORKFLOW_ID`.

In [28]:
WORKFLOW_ID = "workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae"  # Copy from Elastic UI after creating the workflow

In [29]:
# Create the workflow tool
workflow_tool_payload = {
    "id": "run_support_query_router",
    "type": "workflow",
    "description": (
        "Routes a customer support query through the triage workflow. "
        "Searches the knowledge base, classifies query complexity, and "
        "generates a response using the appropriate model. Use this tool "
        "whenever a customer asks a support question."
    ),
    "tags": ["support", "triage", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(response.json())

Workflow tool created: 200
{'id': 'run_support_query_router', 'type': 'workflow', 'description': 'Routes a customer support query through the triage workflow. Searches the knowledge base, classifies query complexity, and generates a response using the appropriate model. Use this tool whenever a customer asks a support question.', 'tags': ['support', 'triage', 'workflow'], 'configuration': {'workflow_id': 'workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae'}, 'readonly': False, 'schema': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'The customer support query'}}, 'required': ['query'], 'additionalProperties': False, 'description': 'Parameters needed to execute the workflow', '$schema': 'http://json-schema.org/draft-07/schema#'}}


In [30]:
# Create the agent with the workflow tool
agent_payload = {
    "id": "support-query-agent",
    "name": "Support Query Agent",
    "description": "Customer support agent that routes queries through a multi-model workflow.",
    "labels": ["support", "e-commerce"],
    "configuration": {
        "instructions": (
            "You are a customer support assistant for BrownBox, an e-commerce platform. "
            "When a customer asks a support question, use the `run_support_query_router` tool "
            "to process it. The tool will search the knowledge base, classify the query, "
            "and generate an appropriate response.\n\n"
            "Present the response to the customer in a friendly, professional tone."
        ),
        "tools": [{"tool_ids": ["run_support_query_router"]}],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

Agent created: 200
{'id': 'support-query-agent', 'type': 'chat', 'name': 'Support Query Agent', 'description': 'Customer support agent that routes queries through a multi-model workflow.', 'labels': ['support', 'e-commerce'], 'configuration': {'instructions': 'You are a customer support assistant for BrownBox, an e-commerce platform. When a customer asks a support question, use the `run_support_query_router` tool to process it. The tool will search the knowledge base, classify the query, and generate an appropriate response.\n\nPresent the response to the customer in a friendly, professional tone.', 'tools': [{'tool_ids': ['run_support_query_router']}]}, 'readonly': False}


### Test the agent

Let's test with queries of varying complexity to see how the workflow routes them.

In [31]:
# Simple query - should match a single FAQ article
simple_converse = {
    "agent_id": "support-query-agent",
    "input": "How do I track my order?",
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=simple_converse,
)
print("Simple query response:")
print(json.dumps(response.json(), indent=2))

Simple query response:
{
  "conversation_id": "e0761a94-c9b0-4d8e-9897-32ee158b428a",
  "steps": [
    {
      "type": "reasoning",
      "reasoning": "The user is asking a support question about tracking their order, so I should use the `run_support_query_router` tool to process it."
    },
    {
      "type": "tool_call",
      "tool_call_id": "tool_run_support_query_router_M7jWHTsLCtu0K6kBOuhl",
      "tool_id": "run_support_query_router",
      "progression": [],
      "params": {
        "query": "How do I track my order?"
      },
      "results": [
        {
          "type": "other",
          "data": {
            "execution": {
              "execution_id": "8d88a128-a1ae-4997-b9e8-cc79a1b275b3",
              "status": "completed",
              "workflow_id": "workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae",
              "started_at": "2026-04-10T18:40:29.696Z",
              "finished_at": "2026-04-10T18:40:47.666Z",
              "output": [
                {
            

In [32]:
# Complex query - requires synthesizing multiple articles
complex_converse = {
    "agent_id": "support-query-agent",
    "input": (
        "I ordered an OTG last week and it arrived damaged. I also noticed "
        "I was charged twice on my credit card. I want a replacement for the "
        "OTG and a refund for the duplicate charge. Also, my account shows "
        "the wrong delivery address - can you update it?"
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=complex_converse,
)
print("Complex query response:")
print(json.dumps(response.json(), indent=2))

Complex query response:
{
  "conversation_id": "b54903e8-4d28-4c48-9937-25283c707f7f",
  "steps": [
    {
      "type": "reasoning",
      "reasoning": "The user is asking a support question about a damaged order, a duplicate charge, and an incorrect delivery address. The `run_support_query_router` tool is designed to handle customer support queries by searching the knowledge base, classifying the query, and generating an appropriate response."
    },
    {
      "type": "tool_call",
      "tool_call_id": "tool_run_support_query_router_zWh2MCW7pk8q0UKzkftk",
      "tool_id": "run_support_query_router",
      "progression": [],
      "params": {
        "query": "I ordered an OTG last week and it arrived damaged. I also noticed I was charged twice on my credit card. I want a replacement for the OTG and a refund for the duplicate charge. Also, my account shows the wrong delivery address - can you update it?"
      },
      "results": [
        {
          "type": "other",
          "data

## Cleanup

In [15]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/support-query-agent", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/run_support_query_router", headers=headers
)
print("Deleted Agent Builder resources")

# Delete custom Mistral connector
requests.delete(
    f"{KIBANA_URL}/api/actions/connector/{MISTRAL_CONNECTOR_ID}", headers=headers
)
print(f"Deleted connector: {MISTRAL_CONNECTOR_ID}")

# Delete knowledge base index
es_client.indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")

Deleted Agent Builder resources
Deleted connector: 4bd3219e-58a5-447f-8284-ac1a94b6e16d
Deleted index: support-knowledge-base
